# RNN & LSTM — From Scratch Implementation

This notebook is my hands-on follow-up to the concept notes and practice questions.
Goal: actually **code** a vanilla RNN and an LSTM using only NumPy (no frameworks),
so the gates/hidden state/cell state stop feeling abstract — then a quick PyTorch
version at the end to see how it looks in a real framework.

**Covers:**
1. Vanilla RNN forward pass (and seeing "fading" happen in real numbers)
2. LSTM forward pass (matching the hand-calculation example from my notes)
3. Same LSTM step, but with PyTorch's `nn.LSTMCell` (sanity check)


In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)


## 1. Vanilla RNN — Forward Pass

Formula: `h(t) = tanh(Wxh · x(t) + Whh · h(t-1) + b)`

Same weights are reused at every timestep (weight sharing) and the hidden state
is always a fixed size — no matter how long the sequence is.


In [2]:
def rnn_cell_forward(x_t, h_prev, Wxh, Whh, b):
    """One RNN timestep."""
    z = Wxh @ x_t + Whh @ h_prev + b
    h_t = np.tanh(z)
    return h_t


def rnn_forward(inputs, h0, Wxh, Whh, b):
    """Run the RNN over a full sequence of inputs."""
    h = h0
    hidden_states = [h]
    for x_t in inputs:
        h = rnn_cell_forward(x_t, h, Wxh, Whh, b)
        hidden_states.append(h)
    return hidden_states


In [3]:
# Tiny example: hidden state size 1, input size 1 (matches the "single number" style
# used in my concept notes, just so the fading is easy to see).

hidden_size = 1
input_size = 1

Wxh = np.array([[0.9]])   # weight on input
Whh = np.array([[0.5]])   # weight on previous hidden state -> this is what drives fading
b = np.array([0.0])

h0 = np.array([0.0])

# A sequence of 10 timesteps, all zeros after the first -> lets us watch
# the influence of the FIRST input fade away, like in the concept notes.
inputs = [np.array([1.0])] + [np.array([0.0])] * 9

hidden_states = rnn_forward(inputs, h0, Wxh, Whh, b)

for t, h in enumerate(hidden_states):
    print(f"t={t:2d}  h(t) = {h[0]:.4f}")


t= 0  h(t) = 0.0000
t= 1  h(t) = 0.7163
t= 2  h(t) = 0.3436
t= 3  h(t) = 0.1701
t= 4  h(t) = 0.0849
t= 5  h(t) = 0.0424
t= 6  h(t) = 0.0212
t= 7  h(t) = 0.0106
t= 8  h(t) = 0.0053
t= 9  h(t) = 0.0026
t=10  h(t) = 0.0013


Notice how quickly `h(t)` shrinks toward 0 even though the very first input was a
full-strength `1.0`. That's the **fading** from my notes, just implemented for real
instead of a hand-wavy example — the repeated multiply-by-`Whh` + `tanh` squashing
is exactly why old information disappears.


## 2. LSTM — Forward Pass

Now the real deal — three gates + a cell state. This directly reproduces the
hand-calculation from my notes (Section E of the practice questions) so I can
check the code against numbers I already verified by hand.

Given: `h(t-1) = 0.5`, `C(t-1) = 0.8`, `x(t) = 1.0`


In [4]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def lstm_cell_forward(x_t, h_prev, C_prev, params):
    """
    One LSTM timestep. `params` is a dict holding W, U, b for each gate:
    forget (f), input (i), candidate (c), output (o).
    """
    Wf, Uf, bf = params["f"]
    Wi, Ui, bi = params["i"]
    Wc, Uc, bc = params["c"]
    Wo, Uo, bo = params["o"]

    f_t = sigmoid(Wf * h_prev + Uf * x_t + bf)      # forget gate: what to erase
    i_t = sigmoid(Wi * h_prev + Ui * x_t + bi)      # input gate: what to add
    C_tilde = np.tanh(Wc * h_prev + Uc * x_t + bc)  # candidate: new content
    C_t = f_t * C_prev + i_t * C_tilde              # new cell state

    o_t = sigmoid(Wo * h_prev + Uo * x_t + bo)      # output gate: what to expose
    h_t = o_t * np.tanh(C_t)                        # new hidden state

    return h_t, C_t, {"f": f_t, "i": i_t, "C_tilde": C_tilde, "o": o_t}


In [5]:
# Same weights as my hand-calculation practice (Section E, Q39-Q45)
params = {
    "f": (0.5, 0.3, 0.1),    # Wf, Uf, bf
    "i": (0.4, 0.6, -0.1),   # Wi, Ui, bi
    "c": (0.3, 0.5, 0.05),   # Wc, Uc, bc
    "o": (0.6, 0.2, 0.0),    # Wo, Uo, bo
}

h_prev = 0.5
C_prev = 0.8
x_t = 1.0

h_t, C_t, gates = lstm_cell_forward(x_t, h_prev, C_prev, params)

print("Forget gate  f(t):", round(gates["f"], 3))     # expect ~0.657
print("Input gate   i(t):", round(gates["i"], 3))     # expect ~0.668
print("Candidate   C~(t):", round(gates["C_tilde"], 3))  # expect ~0.604
print("Cell state   C(t):", round(C_t, 3))            # expect ~0.929
print("Output gate  o(t):", round(gates["o"], 3))     # expect ~0.622
print("Hidden state h(t):", round(h_t, 3))            # expect ~0.455


Forget gate  f(t): 0.657
Input gate   i(t): 0.668
Candidate   C~(t): 0.604
Cell state   C(t): 0.929
Output gate  o(t): 0.622
Hidden state h(t): 0.455


These numbers match the hand calculation exactly (f≈0.657, i≈0.668, C~≈0.604,
C(t)≈0.929, o≈0.622, h(t)≈0.455) — good confirmation the code is doing what the
formulas say.

### Quick experiment: what if the forget gate were smaller?

My notes (Q46) asked: what happens if `f(t) = 0.1` instead of `0.657`, everything
else the same? Let's just run it and see.


In [6]:
h_t_low_f, C_t_low_f, gates_low_f = lstm_cell_forward(
    x_t, h_prev, C_prev,
    params={**params, "f": (0, 0, np.arctan(np.tan(0)))}  # placeholder, overridden below
)

# Simplest way: just plug f(t) = 0.1 directly instead of computing it from weights.
f_t_manual = 0.1
i_t = gates["i"]
C_tilde = gates["C_tilde"]
C_t_manual = f_t_manual * C_prev + i_t * C_tilde
print("If f(t) = 0.1  ->  C(t) =", round(C_t_manual, 3), " (vs. 0.929 when f(t)=0.657)")


If f(t) = 0.1  ->  C(t) = 0.484  (vs. 0.929 when f(t)=0.657)


With a small forget gate, most of the old cell state gets wiped — the network
"decided" the old info isn't worth keeping. With a forget gate close to 1 (like
`0.657` here, or closer to `0.9+` in a well-trained model protecting something
important), the old cell state survives much more intact. This is the whole trick
behind LSTM remembering things like "Ariel" across a long sequence.


## 3. Sanity Check with PyTorch

Just to see how this looks with a real framework instead of hand-rolled NumPy.
`nn.LSTMCell` does the same forget/input/candidate/output math internally —
here I just confirm it runs and produces sensible shapes (weights are randomly
initialized, so the numbers won't match my hand calc, but the mechanics are the
same 3-gate + cell-state structure).


In [7]:
import torch
import torch.nn as nn

torch.manual_seed(0)

input_size = 1
hidden_size = 1

lstm_cell = nn.LSTMCell(input_size, hidden_size)

x_t = torch.tensor([[1.0]])
h_prev = torch.tensor([[0.5]])
C_prev = torch.tensor([[0.8]])

h_t, C_t = lstm_cell(x_t, (h_prev, C_prev))

print("h(t):", h_t.item())
print("C(t):", C_t.item())


h(t): 0.09248969703912735
C(t): 0.24990510940551758


## Takeaways

- Coding the RNN made the "fading" concept concrete — you can literally watch the
  numbers shrink toward zero, timestep by timestep.
- Coding the LSTM and matching it against my hand-calculated answers was the best
  gut-check that I actually understood the formulas, not just memorized them.
- `nn.LSTMCell` in PyTorch does exactly the same forget/input/output gate math,
  just with everything trained instead of fixed — so it was reassuring to see
  the same shapes and structure show up.

**Next step in the roadmap:** NLP basics → attention → Transformers.
